# Auto-Labeling Pipeline — Clean Rebuild (O3 + decoupled B1/B2, multi-dataset, multi-camera)

Rebuilt from `autolabeling_pipeline.ipynb` / `autolabeling/src/autolabeling/`. See
`notes/clean_pipeline_overview.md` for the full design rationale. Summary of what
changed relative to the old pipeline:

- **Dropped**: motion compensation, pointmap modes O1/O2/O4/O5, hull anchoring,
  all OBB output filters (volume/dimension *and* ego-vehicle exclusion — fully
  unfiltered output, on purpose, to see the pipeline's raw shortcomings),
  max-BEV-range LiDAR filter, `BEV_DROP_NON_LOCAL`, cross-camera merge (no
  duplicate suppression is attempted; every camera's detections are shown
  independently).
- **Kept unchanged** (imported directly from the tested `autolabeling` package):
  SAM3 segmentation + cross-class dedup, SAM3D Body inference, B1 depth correction,
  B2 ground anchoring, orientation + OBB fitting, rider merge, LiDAR aggregation +
  ego-body filter, mask erosion.
- **New**: TerraSeg-based ground removal for O3 (before HDBSCAN, before the affine
  fit), an extended O3 fallback ladder (HDBSCAN cluster → raw in-mask points →
  dilated-mask retry → global affine → unscaled), B1/B2 as independent toggles,
  a dataset selector (ECP / nuScenes mini), multi-camera support, intermediate
  visualizations (SAM3 masks, mesh-vs-mask reprojection with real silhouette
  rendering + IoU, interactive 3D LiDAR views), and a BEV view matching the old
  notebook's (LiDAR by height, GT boxes dashed, pseudo-labels solid).

**Dataset + camera model.** One keyframe (= one `sample_token`) is shared across all
its cameras — same LiDAR sweep, same ego pose. So LiDAR aggregation, TerraSeg, and
PseudoLabeler all run **once per keyframe** (camera-independent); SAM3, SAM3D Body,
and SAM3D Objects each load their model **once** and loop over `CAMERAS` before
unloading (cheaper than the production pipeline's per-camera full reload in
`run_multi_camera_pipeline`). No cross-camera duplicate suppression is attempted —
each camera's detections are plotted independently.

This notebook operates on **one keyframe at a time** (exploratory/prototyping
stage). Every stage is written as a function so it can be looped over many
keyframes later, when this gets promoted into the "clean full pipeline".

## 1. Imports

In [ ]:
import os
import sys
import gc
from pathlib import Path

import numpy as np
import torch
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from pyquaternion import Quaternion

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
# Explicitly (re-)enabled, not just left alone: torch.set_grad_enabled() is a
# global flag for the whole kernel process, not per-cell -- if an earlier run in
# this session ever disabled it, merely not calling it again here would NOT undo
# that. PseudoLabeler (Section 7) needs real autograd (it fits its ground-surface
# MLP via backprop); every other stage already wraps itself in its own
# torch.no_grad()/inference_mode() internally where it matters.
torch.set_grad_enabled(True)

## 2. Config

`DEV_ROOT` matches the container mount used by the rest of the codebase
(`/workspace` — same convention as `autolabeling/configs/*.yaml`).

In [ ]:
# ── Paths (dataset-independent) ────────────────────────────────────────────────
DEV_ROOT = '/workspace'

SAM3_CKPT       = f'{DEV_ROOT}/models/SAM3/checkpoints/sam3.pt'
SAM3D_BODY_REPO = f'{DEV_ROOT}/models/SAM3D/sam-3d-body'
SAM3D_OBJ_CFG   = f'{DEV_ROOT}/models/SAM3D/sam-3d-objects/checkpoints/hf/pipeline.yaml'
TERRASEG_CKPT   = f'{DEV_ROOT}/models/TerraSeg/checkpoints/terraseg_s.pth'  # None = auto-download from HF

### 2.1 Dataset + scene/frame + camera selector

Change `DATASET` to switch between ECP and nuScenes mini. ECP is stored in
nuScenes format (same `NuScenes` devkit class, different `dataroot`/`version`),
so the rest of the notebook doesn't need to branch on dataset anywhere else.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Dataset selector  ->  change this one variable to switch datasets
# ══════════════════════════════════════════════════════════════════════════════
DATASET = 'nuscenes_mini'           # 'ecp'  or  'nuscenes_mini'

# ── Per-dataset defaults ───────────────────────────────────────────────────────
if DATASET == 'nuscenes_mini':
    DATA_ROOT    = f'{DEV_ROOT}/data/nuScenes_mini'
    NUSCENES_VER = 'v1.0-mini'
    SCENE_IDX    = 3     # mini has 10 scenes (0-9)
    FRAME_IDX    = 3
    # LiDAR aggregation -- 32-beam sensor -> more sweeps to compensate sparse density
    N_BEFORE = 3
    N_AFTER  = 3
    LIDAR_LINES = 32          # nuScenes: 32-beam Velodyne (not consumed by O3 -- kept for
                              # parity/future O4-O5 experiments, which do need it)
    # All cameras available in nuScenes
    ALL_DATASET_CAMERAS = [
        'CAM_FRONT', 'CAM_FRONT_LEFT', 'CAM_FRONT_RIGHT',
        'CAM_BACK',  'CAM_BACK_LEFT',  'CAM_BACK_RIGHT',
    ]
elif DATASET == 'ecp':
    DATA_ROOT    = f'{DEV_ROOT}/data/ecp'
    NUSCENES_VER = 'v1.0-trainval'
    SCENE_IDX    = 10    # ECP scene (GT available for 10, 11, 15)
    FRAME_IDX    = 560
    # LiDAR aggregation -- 64-beam sensor -> single sweep already dense
    N_BEFORE = 2
    N_AFTER  = 2
    LIDAR_LINES = 64          # ECP: 64-beam LiDAR
    # ECP frontal cameras (left-to-right order)
    ALL_DATASET_CAMERAS = [
        'CAM_FRONT_LEFT', 'CAM_FRONT', 'CAM_FRONT_RIGHT',
    ]
else:
    raise ValueError(f"Unknown DATASET: {DATASET!r}  (must be 'ecp' or 'nuscenes_mini')")

# ── Camera selection ──────────────────────────────────────────────────────────
# CAMERA      : primary / reference camera (used for the TerraSeg aggregation
#               sanity check and as the fallback when ALL_CAMERAS=False)
# ALL_CAMERAS : True  -> run on every camera in ALL_DATASET_CAMERAS
#               False -> run on CAMERA only  (set CAMERA to any single camera you want)
CAMERA      = 'CAM_FRONT'
ALL_CAMERAS = False
CAMERAS     = ALL_DATASET_CAMERAS if ALL_CAMERAS else [CAMERA]
print(f'Dataset: {DATASET}  |  cameras: {CAMERAS}')

### 2.2 Detection thresholds, prompts, HDBSCAN, filters (dataset-independent)

In [ ]:
# ── Detection thresholds ────────────────────────────────────────────────────────
SAM3_SCORE_THRESH = 0.5
MIN_MASK_PX       = 100
BODY_BBOX_THRESH  = 0.8
SAM3_IOU_MERGE    = 0.4

# ── Cross-class dedup (SAM3 masks) ───────────────────────────────────────────────
CROSS_CLASS_IOU_THRESH = 0.5
CROSS_CLASS_PRIORITY = {
    'pedestrian': 0, 'car': 1, 'bicycle': 2, 'truck': 3,
    'motorcycle': 4, 'trailer': 5, 'bus': 6, 'construction vehicle': 7,
}

# ── Prompts: 'body' -> SAM3D Body | 'objects' -> SAM3D Objects ──────────────────
PROMPTS = {
    'pedestrian': 'body',
    'car': 'objects', 'bicycle': 'objects', 'motorcycle': 'objects',
    'bus': 'objects', 'truck': 'objects',
    'construction vehicle': 'objects', 'trailer': 'objects',
}

# ── LiDAR pre-filter (ego-body exclusion only -- range/volume/dim filters dropped) ──
EGO_BOX_HALF_X = 4.0
EGO_BOX_HALF_Y = 1.5
EGO_BOX_Z_MIN  = 0.5
EGO_BOX_Z_MAX  = 2.5

# ── HDBSCAN per-class params (shared by B1 + O3) ─────────────────────────────────
HDBSCAN_PARAMS = {
    'pedestrian':           dict(min_cluster_size=3, min_samples=1, cluster_eps=0.20),
    'bicycle':              dict(min_cluster_size=3, min_samples=1, cluster_eps=0.20),
    'motorcycle':           dict(min_cluster_size=3, min_samples=1, cluster_eps=0.30),
    'car':                  dict(min_cluster_size=3, min_samples=1, cluster_eps=0.50),
    'truck':                dict(min_cluster_size=5, min_samples=1, cluster_eps=0.60),
    'construction vehicle': dict(min_cluster_size=5, min_samples=1, cluster_eps=0.60),
    'bus':                  dict(min_cluster_size=5, min_samples=1, cluster_eps=0.80),
    'trailer':              dict(min_cluster_size=5, min_samples=1, cluster_eps=0.70),
}

# ── B1 / B2 -- independent toggles (old pipeline hard-coupled them) ──────────────
USE_B1 = True   # depth (tz) correction from LiDAR
USE_B2 = True   # ground anchoring via PseudoLabeler

# ── Sweep aggregation on/off ──────────────────────────────────────────────────
# False -> both pts_ego_full (Section 5, PseudoLabeler) and pts_ego (Section 6,
# TerraSeg-per-sweep then aggregated, B1 + O3) fall back to the single anchor
# sweep only. N_BEFORE/N_AFTER (Section 2.1) are ignored either way when False.
USE_AGGREGATION = True

# ── O3 -- mask erosion, fallback ladder ───────────────────────────────────────────
# Ground removal is no longer an O3-local option: pts_ego (Section 6) is already
# ground-free by construction (TerraSeg runs per sweep before aggregation), so
# there is nothing left to toggle here.
O3_MASK_ERODE_PX  = 3     # erode mask before HDBSCAN point selection (0 = off)
O3_MIN_AFFINE_PTS = 4
# A mask-dilation retry tier (local_dilated / local_raw_dilated, between
# local_raw and global_fallback) existed at one point -- removed for now.

# No OBB output filter at all in this rebuild (not even ego-vehicle exclusion) --
# we want fully unfiltered output to see the pipeline's raw shortcomings.

# ── Pointmap modes to run (Section 10) -- a LIST: every mode is run on the same masks/objects and compared ──
# 1: sparse_lidar          -- raw LiDAR only, NaN wherever there is no direct return.
# 2: moge_affine_local     -- MoGe + per-object local affine fit. Outside the mask the map is the same MoGe scene
#                              with THIS object's affine applied everywhere.
# 3: moge_affine_local_masked -- as 2, but NaN outside the mask.
# 4: completionformer_full -- CompletionFormer dense depth over the whole image.
# 5: moge_affine_local_raw -- as 2, fitted on ALL in-mask points (no erosion, no HDBSCAN).
# 6: moge_affine_local_piecewise -- object affine inside the mask + a separate robust affine for the background.
# 7: moge_affine_composite -- ONE map: every mask its own affine, sectioned background affine.
# 8: moge_affine_local_regslope -- mode 6 + ill-conditioned object slopes replaced by the background slope.
# 9: completionformer_raw -- as 4, but with NO per-mask HDBSCAN cleaning: raw ground-free LiDAR goes in as-is.
# 10: completionformer_ground -- as 4, but in-mask anchors from the GROUND-FREE cloud + background
#                  anchors from the FULL cloud (ground included) -- mode 6's object/background split,
#                  applied to CompletionFormer's anchors.
# 11: ldcm_full -- same hybrid anchors as mode 10, but fused by LDCM (frozen MoGe prior + learned
#                  per-pixel LiDAR fusion) instead of CompletionFormer. Needs LDCM_REPO/LDCM_CKPT/
#                  LDCM_MOGE_CKPT set below.
POINTMAP_MODES_TO_RUN = [11]
MESH_POINTS = 20000      # the OBB is fitted on the FULL mesh first; then only this many vertices are kept (bicycles/motorcycles keep all)

# ── Token-drop probe (runs in Section 10, analysed in Section 11.6) ───────────────────────────────────────
# Re-runs PROBE_MODE with parts of SAM3D's pointmap conditioning zeroed, to measure how much the trained network
# actually uses the pointmap content. The shift/scale SAM3D derives from the map are NOT affected.
#   'repeat'    normal again -- must reproduce the normal run exactly (seed is fixed): checks determinism
#   'drop_both' both pointmap streams zeroed (the training-time dropout did exactly this, 10% of the time)
#   'drop_full' only the full-image stream    'drop_crop' only the crop stream  (single streams were never dropped in training)
# Costs about 17 s per object per run.
RUN_TOKEN_PROBE = False
PROBE_MODE = 2
PROBE_VARIANTS = {'repeat': None, 'drop_both': ['pointmap', 'rgb_pointmap'], 'drop_full': ['rgb_pointmap'], 'drop_crop': ['pointmap']}
POINTMAP_MODE_NAMES = {
    1: 'sparse_lidar',
    2: 'moge_affine_local',
    3: 'moge_affine_local_masked',
    4: 'completionformer_full',
    5: 'moge_affine_local_raw',   # same fit as 2 on ALL in-mask points: no erosion, no HDBSCAN
    6: 'moge_affine_local_piecewise',   # object affine in the mask + robust background affine outside
    7: 'moge_affine_composite',   # ONE map: every mask its own affine, sectioned background affine
    8: 'moge_affine_local_regslope',   # mode 6 + ill-conditioned object slopes replaced by the background slope
    9: 'completionformer_raw',   # as 4, but NO per-mask HDBSCAN cleaning: raw ground-free LiDAR as-is
    10: 'completionformer_ground',   # ground-free in-mask anchors + full-cloud background (mode 6's split)
    11: 'ldcm_full',   # same hybrid anchors as mode 10, fused by LDCM instead of CompletionFormer
}
CFORMER_CKPT = f'{DEV_ROOT}/models/CompletionFormer/checkpoints/KITTIDC/KITTIDC_L1L2.pt'  # modes 4/9/10
# Modes 4/10/11: on a mask where HDBSCAN finds no dominant cluster (too few points, or no cluster at
# all), its in-mask points are KEPT unfiltered by default. Set to True to drop them instead so the
# model completes from image priors + out-of-mask anchors only -- flip and rerun Section 10 to compare.
# (mode 9 always skips this cleaning step, so the flag has no effect there.)
CFORMER_DROP_ON_FAILURE = False
# Mode 11 only -- github.com/aigc3d/LDCM, cloned + downloaded per the commands given alongside this
# notebook. ldcm_repo is the cloned repo ROOT (contains the `ldcm/` package). ldcm_ckpt is LDCM's own
# checkpoint (a local dir/file or a bare HF repo id -- LDCM's own code resolves a dir to <dir>/ldcm.pt).
# ldcm_moge_ckpt is its bundled MoGe prior's weights -- unlike ldcm_ckpt this one is NOT dir-resolved,
# so it must be the .pt FILE itself (or a bare HF repo id).
LDCM_REPO      = f'{DEV_ROOT}/models/LDCM'
LDCM_CKPT      = f'{DEV_ROOT}/models/LDCM/checkpoints/LDCM'
LDCM_MOGE_CKPT = f'{DEV_ROOT}/models/LDCM/checkpoints/moge-2-vits-normal/model.pt'
# LDCM's bundled MoGe code calls utils3d.pt.*; the utils3d already in this container only has the
# same functions under utils3d.torch (checked: identical signatures, just a naming difference between
# utils3d releases). This points at a `pip install --target=<dir> utils3d` of the current utils3d so
# _load_ldcm can alias .pt = .torch -- only loaded for mode 11, nothing else is affected.
LDCM_UTILS3D_VENDOR = '/home/nfs/lleba/.ldcm_utils3d_vendor'

## 3. Package import

Reusing the tested `autolabeling` package for everything we're keeping unchanged
(SAM3 segmentation, SAM3D Body, B1/B2 helpers, orientation/OBB, LiDAR utils). Only
the O3 pointmap logic gets a new implementation below, since that's where ground
removal + the fallback ladder are new. `ecp_loader` re-exports the exact same
loading functions as `nuscenes_loader` (ECP is stored in nuScenes format), so we
only ever need to import from `nuscenes_loader`.

In [ ]:
_SRC = f'{DEV_ROOT}/autolabeling/src'
if _SRC not in sys.path:
    sys.path.insert(0, _SRC)

from autolabeling.data.nuscenes_loader import collect_frames
from autolabeling.utils.lidar import (
    load_lidar_pts_aggregated, load_lidar_pts_nonground_aggregated,
    filter_inmask_lidar_hdbscan, project_lidar_to_camera,
)
from autolabeling.utils.geometry import cam_to_ego, ego_to_global, project, _BBOX_EDGES
from autolabeling.models.sam3_segmentor import SAM3Segmentor
from autolabeling.models.sam3d_body import SAM3DBodyModel
from autolabeling.models.sam3d_objects import SAM3DObjectsModel, resolve_pointmap_mode
from autolabeling.utils.meshes import attach_full_obbs, slim_object_results
from autolabeling.utils.diagnostics import gt_boxes_ego, associate_gt, associate_gt_lidar, object_errors, resolve_duplicate_gt
from autolabeling.fitting.obb import compute_obb_pedestrian, compute_obb_gravity_aligned, disambiguate_yaw
from autolabeling.orientation.pedestrian import facing_direction
from autolabeling.utils.terraseg import TerraSegGroundFilter
from autolabeling.pipeline import (
    _apply_b1_depth_correction, _apply_b2_ground_anchoring, _fit_pseudolabeler,
    _merge_rider_obbs, _postprocess_frame,
)

print('autolabeling package imported OK.')

### 3.1 Shared helper — class colors

Mesh-vs-mask silhouette rendering (previously here) has been removed -- it was
purely diagnostic (fed no downstream pipeline output), and the pytorch3d
rendering/open3d decimation it needed proved fragile enough to defer to a
dedicated follow-up notebook once the main pipeline is validated.

In [ ]:
_PROMPT_COLORS = {
    'pedestrian': (255, 80, 80), 'car': (80, 160, 255), 'bicycle': (80, 220, 120),
    'motorcycle': (255, 200, 60), 'bus': (200, 120, 255), 'truck': (255, 120, 200),
    'trailer': (150, 150, 255), 'construction vehicle': (255, 255, 120),
}

## 4. Data loading

Loads the same keyframe (`SCENE_IDX`, `FRAME_IDX`) for every camera in `CAMERAS`.
All of them share the same LiDAR sweep and ego pose -- only intrinsics/extrinsics
and the image differ per camera.

In [ ]:
from nuscenes.nuscenes import NuScenes

nusc = NuScenes(version=NUSCENES_VER, dataroot=DATA_ROOT, verbose=False)
scene_name = nusc.scene[SCENE_IDX]['name']
print(f'Scene: {scene_name}')

frames = {
    cam: collect_frames(nusc, [scene_name], cam, FRAME_IDX, FRAME_IDX + 1)[0]
    for cam in CAMERAS
}
for cam, f in frames.items():
    print(f'  {cam}: {f.scene_name} #{f.frame_idx}  ({f.img_width}x{f.img_height})')

n_show = len(CAMERAS)
fig, axes = plt.subplots(1, n_show, figsize=(6 * n_show, 4.5), squeeze=False)
for ax, cam in zip(axes[0], CAMERAS):
    img_rgb, _ = frames[cam].load_images()
    ax.imshow(img_rgb); ax.axis('off'); ax.set_title(cam)
plt.suptitle(f'{scene_name}  frame {FRAME_IDX}')
plt.tight_layout();

## 5. LiDAR pre-processing — full aggregation (`pts_ego_full`, for PseudoLabeler)

Camera-independent: aggregation only depends on the keyframe's `lidar_sd_token` /
ego pose, identical for every camera of this sample -- runs **once**. (There is a
second, unrelated "ego" filter concept -- output *OBB* filtering -- which this
rebuild doesn't use at all, see Section 2.)

Standard multi-sweep aggregation (ground **included**) + the ego-body point
filter (removes LiDAR points hitting the ego vehicle's own roof/mount, on the raw
cloud, before any model runs). **This full cloud feeds only PseudoLabeler**
(Section 7) and B2's RANSAC fallback -- both need actual ground points to find
where the ground is. B1 and O3 use a *different*, ground-free cloud built in
Section 6 -- see that section for why the two are built differently, not just
filtered differently.

In [ ]:
EGO_FILTER_KW = dict(
    use_ego_body_filter=True,
    ego_box_half_x=EGO_BOX_HALF_X, ego_box_half_y=EGO_BOX_HALF_Y,
    ego_box_z_min=EGO_BOX_Z_MIN, ego_box_z_max=EGO_BOX_Z_MAX,
)

# ref_frame can be any camera's FrameRecord for this keyframe -- LiDAR fields are
# identical across cameras of the same sample. Same function the production pipeline
# uses; the ego-body filter runs per sweep in that sweep's own frame. n=0 -> one sweep.
_n_before_eff = N_BEFORE if USE_AGGREGATION else 0
_n_after_eff  = N_AFTER if USE_AGGREGATION else 0

pts_ego_full = load_lidar_pts_aggregated(nusc, frames[CAMERA], _n_before_eff, _n_after_eff, **EGO_FILTER_KW)
print(f'Full aggregated + ego-filtered point cloud: {len(pts_ego_full):,} points  '
      f'(PseudoLabeler + B2 RANSAC fallback only)')

## 6. TerraSeg ground removal — per sweep, before aggregation (`pts_ego`)

**Ground removal happens before aggregation, on each sweep independently, in
that sweep's own native ego frame.** This is a deliberate decision, written up in
full in `notes/clean_pipeline_overview.md` (for when it needs defending later) --
summary:

TerraSeg (PTv3, trained on OmniLiDAR) was trained and evaluated strictly on
single, independent scans, with no multi-frame accumulation. Its input features
(normalized height, horizontal range) are defined relative to each sweep's own
sensor origin. Running it once on an already-aggregated cloud would be
out-of-distribution, would compute range/height for non-anchor points relative
to the *wrong* origin (the anchor's, not their own sweep's), and would expose the
model to motion-smeared trails from dynamic objects.

So: for each sweep, load it in its own ego frame, apply the ego-body point filter
there (same reasoning as always -- avoids the displacement artefact of filtering
after merging different timestamps), classify with TerraSeg, keep only the
non-ground points, *then* transform those into the anchor frame and concatenate.
Non-ground labels propagate through the ego-motion transform "for free" -- the
resulting aggregated cloud (`pts_ego`) is ground-free at no extra cost, with no
separate ground/non-ground mask needed downstream (Section 10's O3 candidate
selection no longer needs a `non_ground_vis` argument at all).

This is the mirror image of Section 5's PseudoLabeler cloud: PseudoLabeler
*wants* the full aggregated cloud including ground, because as an offline
per-frame optimization it benefits from the extra density rather than being hurt
by an unfamiliar input distribution.

`USE_AGGREGATION` (Section 2.2) gates this too: when `False`, only the anchor
sweep is walked (`n_before=n_after=0`) -- TerraSeg still runs per-sweep as
described above, just on a single sweep, degenerating cleanly to no
aggregation.

In [ ]:
terraseg_filter = TerraSegGroundFilter(Path(DEV_ROOT), variant='S', ckpt_path=TERRASEG_CKPT)

# Per sweep, in that sweep's own ego frame: ego-body filter -> TerraSeg -> keep non-ground
# -> transform to the anchor frame -> concatenate. (Shared with the production pipeline.)
pts_ego = load_lidar_pts_nonground_aggregated(
    nusc, frames[CAMERA], _n_before_eff, _n_after_eff, terraseg_filter, **EGO_FILTER_KW)
print(f'Ground-free aggregated point cloud: {len(pts_ego):,} points  (B1 + O3 only)')

terraseg_filter.unload()
gc.collect(); torch.cuda.empty_cache()

proj = {}
for cam in CAMERAS:
    pts_ego_vis, u_vis, v_vis, Z_vis, H, W = project_lidar_to_camera(frames[cam], pts_ego)
    proj[cam] = dict(pts_ego_vis=pts_ego_vis, u_vis=u_vis, v_vis=v_vis, Z_vis=Z_vis, H=H, W=W)
    print(f'  {cam}: {len(pts_ego_vis):,} visible (ground-free)')

## 7. PseudoLabeler ground-surface fit

Only needed when `USE_B2=True` — B2 queries this fitted surface at each
pedestrian's (x, y) for the ground-anchoring shift. Fit on `pts_ego_full`
(Section 5, ground included, fully aggregated) -- deliberately *not* `pts_ego`
(Section 6), which has no ground points left to fit against. Camera-independent,
so this also runs once per keyframe, reused by every camera.

In [ ]:
pl_model = None
if USE_B2:
    pl_model = _fit_pseudolabeler(pts_ego_full, DEVICE, Path(DEV_ROOT))
    print('PseudoLabeler fitted.' if pl_model is not None else 'PseudoLabeler fit failed -- B2 will fall back to RANSAC.')

## 8. SAM3 segmentation

In [ ]:
segmentor = SAM3Segmentor(
    checkpoint_path=SAM3_CKPT, prompts=PROMPTS,
    score_thresh=SAM3_SCORE_THRESH, min_mask_px=MIN_MASK_PX, device=DEVICE,
    cross_class_iou_thresh=CROSS_CLASS_IOU_THRESH, cross_class_priority=CROSS_CLASS_PRIORITY,
)
segmentor.load()

sam3_results = {}
for cam in CAMERAS:
    sam3_results[cam] = segmentor.run_frame(frames[cam])
    n = {p: len(d) for p, d in sam3_results[cam].items() if d}
    print(f'  {cam}: {n}')

segmentor.unload()
gc.collect(); torch.cuda.empty_cache()

### 8.1 SAM3 masks with detection numbers

Every detection gets a number (`#id` on its mask, listed in the table below). SAM3D
Body / Objects are the slow part, so Section 8.2 lets you pick which numbers to run.

In [ ]:
DETS = []   # one row per SAM3 detection, numbered across cameras/prompts
for cam in CAMERAS:
    for prompt, dets in sam3_results[cam].items():
        for i, d in enumerate(dets):
            DETS.append(dict(id=len(DETS) + 1, cam=cam, prompt=prompt, idx=i, det=d,
                             stage=PROMPTS[prompt], area=int(d['binary_mask'].sum())))

n_show = len(CAMERAS)
fig, axes = plt.subplots(1, n_show, figsize=(7 * n_show, 5), squeeze=False)
for ax, cam in zip(axes[0], CAMERAS):
    img_rgb_cam, _ = frames[cam].load_images()
    overlay = img_rgb_cam.copy().astype(np.float32)
    for r in DETS:
        if r['cam'] != cam:
            continue
        color = np.array(_PROMPT_COLORS.get(r['prompt'], (200, 200, 200)), dtype=np.float32)
        m = r['det']['binary_mask']
        overlay[m] = 0.5 * overlay[m] + 0.5 * color
    ax.imshow(overlay.astype(np.uint8)); ax.axis('off'); ax.set_title(cam)
    for r in DETS:
        if r['cam'] != cam:
            continue
        ys, xs = np.where(r['det']['binary_mask'])
        ax.text(xs.mean(), ys.mean(), str(r['id']), color='white', fontsize=11, fontweight='bold',
                ha='center', va='center',
                bbox=dict(boxstyle='circle,pad=0.2', fc='black', ec='white', lw=1))
plt.suptitle('SAM3 segmentation masks (numbers = detection id)')
plt.tight_layout()

print(f'{"id":>3}  {"camera":<16} {"class":<22} {"stage":<8} {"score":>5} {"area_px":>8}')
for r in DETS:
    print(f'{r["id"]:>3}  {r["cam"]:<16} {r["prompt"]:<22} {r["stage"]:<8} '
          f'{r["det"]["score"]:>5.2f} {r["area"]:>8}')

### 8.2 Choose what to run

`SELECT_IDS = None` runs everything; a list such as `[3, 7]` runs SAM3D Body / Objects
only for those detections (Body for `body`-stage ids, Objects for `objects`-stage ids;
everything else is skipped). Nothing else changes -- an object gets the same
pointmap it would in a full run (mode 4 still uses all masks for anchor cleaning).
With a selection, ViTDet-only pedestrians (no SAM3 mask) are dropped.

In [ ]:
SELECT_IDS = None      # None = all detections, or e.g. [3, 7]

_ids = {r['id'] for r in DETS}
if SELECT_IDS is not None:
    _bad = sorted(set(SELECT_IDS) - _ids)
    assert not _bad, f'unknown detection id(s) {_bad}; valid: 1..{len(DETS)}'

# SEL[cam][prompt] = set of indices into sam3_results[cam][prompt] to run
SEL = {cam: {} for cam in CAMERAS}
for r in DETS:
    if SELECT_IDS is None or r['id'] in SELECT_IDS:
        SEL[r['cam']].setdefault(r['prompt'], set()).add(r['idx'])

_chosen = [r for r in DETS if SELECT_IDS is None or r['id'] in SELECT_IDS]
print(f'Running {len(_chosen)} of {len(DETS)} detection(s): '
      + ', '.join(f'#{r["id"]} {r["prompt"]} ({r["cam"]})' for r in _chosen))

## 9. SAM3D Body + B1 (depth correction) + B2 (ground anchoring)

SAM3D Body only sees a bounding box (`use_mask=False` inside the wrapper) — the
2D mask is never a constraint on its output. Its depth (`tz`) and its own
regressed body height are coupled (classic monocular height-depth ambiguity), so:

- **B1** overrides `tz` from LiDAR (HDBSCAN median depth) and rigid-shifts the
  mesh — fixes the depth-scale ambiguity, but only pins the mask *centroid*, not
  the full vertical extent, and doesn't rescale the mesh.
- **B2** separately snaps the mesh's lowest vertex to the local ground height —
  fixes the residual height/shape error B1's translation-only correction can't
  touch.

They're independent toggles here (the old pipeline hard-coupled them) so their
individual contribution can actually be measured. Model loaded once, looped over
`CAMERAS`.

In [ ]:
# SAM3 loading leaves BF16 autocast globally enabled, which breaks SAM3D Body.
if torch.is_autocast_enabled():
    torch.autocast('cuda', enabled=False).__enter__()

body_model = SAM3DBodyModel(
    repo_path=SAM3D_BODY_REPO,
    checkpoint_path=f'{SAM3D_BODY_REPO}/checkpoints/sam-3d-body-dinov3/model.ckpt',
    mhr_path=f'{SAM3D_BODY_REPO}/checkpoints/sam-3d-body-dinov3/assets/mhr_model.pt',
    bbox_thresh=BODY_BBOX_THRESH, iou_merge_thresh=SAM3_IOU_MERGE, device=DEVICE,
)
body_model.load()

body_results = {}
for cam in CAMERAS:
    ped_dets = sam3_results[cam].get('pedestrian', [])
    if SELECT_IDS is not None:
        ped_dets = [d for i, d in enumerate(ped_dets) if i in SEL[cam].get('pedestrian', set())]
        if not ped_dets:
            body_results[cam] = []
            print(f'  {cam}: no pedestrian selected, skipped')
            continue
    result = body_model.run_frame(frames[cam], ped_dets)
    if SELECT_IDS is not None:
        result = [r for r in result if r.get('sam3_mask_idx') is not None]   # drop ViTDet-only extras

    if USE_B1 and result and frames[cam].lidar_path is not None:
        p = proj[cam]
        _apply_b1_depth_correction(
            result, frames[cam], ped_dets,
            p['pts_ego_vis'], p['u_vis'], p['v_vis'], p['Z_vis'], p['H'], p['W'],
            hdbscan_kwargs=HDBSCAN_PARAMS.get('pedestrian'),
        )

    if USE_B2 and result:
        # pts_ego_full, not pts_ego -- B2's RANSAC fallback needs actual ground
        # points to fit a plane against; pts_ego has none left.
        _apply_b2_ground_anchoring(result, frames[cam], pts_ego_full, pl_model, DEVICE)

    body_results[cam] = result
    print(f'  {cam}: {len(result)} pedestrian(s)')

body_model.unload()
gc.collect(); torch.cuda.empty_cache()

## 10. SAM3D Objects — one or several pointmap modes (`POINTMAP_MODES_TO_RUN`)

Every mode in the list is run on the **same SAM3 masks and the same selected objects** (Section 8.2), so results can
be compared object by object. The model is loaded once; a mode only changes which pointmap SAM3D is handed.

| Mode | What SAM3D sees |
|---|---|
| 1 `sparse_lidar` | raw LiDAR only, NaN wherever there is no return |
| 2 `moge_affine_local` | MoGe depth calibrated with an affine fitted on this object's cleaned in-mask LiDAR, applied to the whole image |
| 3 `moge_affine_local_masked` | as 2, but NaN outside the object's mask |
| 4 `completionformer_full` | CompletionFormer dense depth over the whole image |
| 5 `moge_affine_local_raw` | as 2, fitted on ALL in-mask points (no erosion, no HDBSCAN) |
| 6 `moge_affine_local_piecewise` | object affine inside the mask + a separate robust affine for the background |
| 7 `moge_affine_composite` | ONE map: every mask its own affine, sectioned background affine |
| 8 `moge_affine_local_regslope` | mode 6 + ill-conditioned object slopes replaced by the background slope |
| 9 `completionformer_raw` | as 4, but NO per-mask HDBSCAN cleaning: raw ground-free LiDAR as-is |
| 10 `completionformer_ground` | ground-free in-mask anchors + full-cloud (ground included) background anchors |
| 11 `ldcm_full` | same hybrid anchors as mode 10, fused by LDCM (frozen MoGe prior + learned per-pixel LiDAR fusion) instead of CompletionFormer |

Modes 4/10/11 (`CFORMER_DROP_ON_FAILURE`): on a mask where HDBSCAN finds no dominant cluster, they keep its in-mask
points unfiltered by default. Set `CFORMER_DROP_ON_FAILURE = True` and rerun this section to drop them instead, so
the model completes from image priors + out-of-mask anchors only. Mode 9 always skips this
cleaning step entirely (raw LiDAR in), so the flag has no effect there. Mode 4 uses the ground-free cloud for
everything (in-mask cleaned + out-of-mask as-is). Modes 10/11 use the SAME cleaned in-mask anchors as mode 4 (still
from the ground-free cloud, so a mask that dips to the road can't pull ground points into the object's own cluster),
but their background anchors come from the FULL cloud instead (ground included) -- the same object/background split
mode 6 already uses between its object fit and background affine. Mode 11's only difference from mode 10 is which
network fuses those anchors with the image (LDCM instead of CompletionFormer) -- see
`notes/sam3d_objects_mode_decision.md` section on why LDCM specifically.

Each result also records SAM3D's own **scale and shift** for that object (`ssi_scale`, `ssi_shift`):
the network predicts size/offset in normalized units and SAM3D decodes `metres = predicted * scale + shift`;
shift = median 3D point inside the mask, scale = a whole-map statistic. They are compared in Section 11.5.

**Token-drop probe** (`RUN_TOKEN_PROBE`): after the modes, `PROBE_MODE` is re-run with parts of the pointmap tokens
zeroed (`PROBE_VARIANTS`); Section 11.6 shows how much the result changes.

In [ ]:
obj_model = SAM3DObjectsModel(
    repo_path=str(Path(SAM3D_OBJ_CFG).parent.parent.parent),
    config_path=SAM3D_OBJ_CFG, prompts=PROMPTS, device=DEVICE,
    pointmap_mode=POINTMAP_MODES_TO_RUN[0],
    min_affine_pts=O3_MIN_AFFINE_PTS,
    hdbscan_params=HDBSCAN_PARAMS,
    cformer_ckpt=CFORMER_CKPT, lidar_lines=LIDAR_LINES,
    cformer_drop_on_failure=CFORMER_DROP_ON_FAILURE,
    ldcm_repo=LDCM_REPO, ldcm_ckpt=LDCM_CKPT, ldcm_moge_ckpt=LDCM_MOGE_CKPT,
    ldcm_utils3d_vendor=LDCM_UTILS3D_VENDOR,
    mask_erode_px=O3_MASK_ERODE_PX, mask_erode_min_px=0,
)
obj_model.load()

# Same code path as the production pipeline (SAM3DObjectsModel.run_frame), once per run.
# A run = a pointmap mode, plus (token-drop probe) a list of conditioning inputs whose tokens are zeroed.
RUNS = {m: (m, None) for m in POINTMAP_MODES_TO_RUN}                 # label -> (mode number, dropped inputs)
if RUN_TOKEN_PROBE:
    if PROBE_MODE not in RUNS:
        RUNS[PROBE_MODE] = (PROBE_MODE, None)
    for vname, drop in PROBE_VARIANTS.items():
        RUNS[str(PROBE_MODE) + ':' + vname] = (PROBE_MODE, drop)

obj_results_by_mode = {}                                              # keyed by run label
for label, (mode_no, drop) in RUNS.items():
    obj_model.pointmap_mode = resolve_pointmap_mode(mode_no)
    cformer_modes = ('completionformer_full', 'completionformer_raw', 'completionformer_ground')
    if obj_model.pointmap_mode in cformer_modes and obj_model._cformer is None:
        obj_model._load_cformer()
    elif obj_model.pointmap_mode == 'ldcm_full' and obj_model._ldcm is None:
        obj_model._load_ldcm()
    obj_model.set_force_drop(drop)
    print('\n=== run ' + str(label) + ': pointmap mode ' + str(mode_no) + ' (' + POINTMAP_MODE_NAMES[mode_no] + ')' +
          (' | zeroed tokens: ' + str(drop) if drop else '') + ' ===')
    per_cam = {}
    for cam in CAMERAS:
        only = None
        if SELECT_IDS is not None:
            only = {(pr, i) for pr, idxs in SEL[cam].items() if PROMPTS[pr] == 'objects' for i in idxs}
            if not only:
                per_cam[cam] = []
                print('  ' + cam + ': no object selected, skipped')
                continue
        res = obj_model.run_frame(frames[cam], sam3_results[cam], pts_ego, only=only,
                                  pts_ego_full=pts_ego_full)   # full cloud: modes 6/7/8 background fit, mode 10 input
        attach_full_obbs(res, frames[cam])                    # OBB from the FULL mesh (used in Section 11)
        per_cam[cam] = slim_object_results(res, MESH_POINTS)   # then reduce what is kept in memory
        print('  ' + cam + ': ' + str(len(res)) + ' object(s)')
    obj_results_by_mode[label] = per_cam
obj_model.set_force_drop(None)

obj_model.unload()
gc.collect(); torch.cuda.empty_cache()

## 11. Orientation estimation + OBB fitting + rider merge (once per mode)

In [ ]:
import copy

results_by_mode = {}          # mode -> {'body': {cam: [...]}, 'objs': {cam: [...]}}
for mode_no in obj_results_by_mode:               # modes, and (if enabled) the token-drop probe runs
    body_m = copy.deepcopy(body_results)          # pedestrians do not depend on the mode
    objs_m = copy.deepcopy(obj_results_by_mode[mode_no])
    for cam in CAMERAS:
        _postprocess_frame(frames[cam], body_m[cam], objs_m[cam])
        body_m[cam], objs_m[cam] = _merge_rider_obbs(frames[cam], body_m[cam], objs_m[cam])
    results_by_mode[mode_no] = dict(body=body_m, objs=objs_m)
    print('run ' + str(mode_no) + ': ' +
          ', '.join(cam + ': ' + str(len(body_m[cam])) + ' pedestrian(s), ' + str(len(objs_m[cam])) + ' object(s)'
                    for cam in CAMERAS))

## 11.5 Diagnostics — how good is each mode, object by object?

Which GT box belongs to a detected object? The LiDAR returns that fall on the object's mask pixels lie on the
object's visible surface, so the GT box (3D) that **contains most of them** is that object. Objects that overlap in the
image sit at different depths and therefore in different 3D boxes, so occlusion does not fool this. The choice does **not**
depend on the pointmap mode, so all modes are scored against the *same* GT box. Not compared (and listed): objects with too
few LiDAR points on the mask or in any box, and ambiguous ones (a second box holds at least half as many points).
A single LiDAR sweep is used (multi-sweep clouds smear moving cars). Caveat: only objects with LiDAR returns can be scored,
which favours nearer, better-lit objects; set `USE_MASK_OVERLAP_FALLBACK = True` to also match the rest by 2D mask overlap
(less reliable).

| column | meaning |
|---|---|
| `center_err` | BEV distance to the GT center [m] |
| `range_err` | error along the ray from the ego vehicle to the object [m]; **+ = predicted too far** (a depth error) |
| `lat_err` | error across that ray [m] |
| `long_ratio`, `short_ratio`, `h_ratio` | predicted / GT footprint long side, short side, height (1.0 = perfect) |
| `yaw_err` | heading error [deg], 0–90; a front/back flip is not counted |
| `scale`, `shift_z` | SAM3D's numbers for this object's pointmap: whole-map scale [m], and depth of the shift point [m] (median depth inside the mask) |
| `obj_slope` | `a` of the object's own affine `Z = a*Zmoge + b` (about 6-9 in a healthy fit of these scenes; near 0 or negative = ill-conditioned; mode 8 replaces such slopes by the background slope) |

**How to read it.** Large `range_err` with small `lat_err` = a depth problem (compare `shift_z` between modes).
Ratios consistently off = a size problem (compare `scale`). `yaw_err` is mostly decided by the image, not the pointmap.
With only a handful of objects this is anecdotal: use it to check that a mode works and to see *why* it differs,
then decide on the batch over many scenes.

In [ ]:
import pandas as pd
from IPython.display import display

gt_boxes = gt_boxes_ego(nusc, frames[CAMERA].sample_token, frames[CAMERA].R_e2g, frames[CAMERA].t_e2g)
print('GT: ' + str(len(gt_boxes)) + ' annotation(s) for this keyframe.')
DET_ID = {(r['cam'], r['prompt'], r['idx']): r['id'] for r in DETS}     # the #id printed on the masks in 8.1

# GT classes that count as 'objects' (the ones SAM3D Objects handles): nuScenes 'vehicle.*'; ECP has flat class names
GT_CATEGORIES = ('vehicle',) if DATASET == 'nuscenes_mini' else ('car', 'trailer', 'motorcycle', 'bicycle')
pts_single = load_lidar_pts_aggregated(nusc, frames[CAMERA], 0, 0, **EGO_FILTER_KW)   # ONE sweep: no smearing
USE_MASK_OVERLAP_FALLBACK = False   # True: objects without a LiDAR match are matched by 2D mask overlap (less reliable)

assoc = {}                    # (cam, prompt, idx) -> (gt index or None, how, ambiguous, mask px); independent of the mode
for res in results_by_mode.values():
    for cam in CAMERAS:
        for r in res['objs'][cam]:
            key = (cam, r['prompt'], r['sam3_index'])
            if key in assoc:
                continue
            mask_k = sam3_results[cam][r['prompt']][r['sam3_index']]['binary_mask']
            gi, frac, amb = associate_gt_lidar(gt_boxes, mask_k, frames[cam], pts_single, category_prefix=GT_CATEGORIES)
            how = 'lidar'
            if gi is None and USE_MASK_OVERLAP_FALLBACK:
                gi, _ = associate_gt(gt_boxes, mask_k, frames[cam], category_prefix=GT_CATEGORIES)
                how, amb = 'mask', False
            assoc[key] = (gi, how, amb, int(mask_k.sum()))
# one GT box belongs to ONE detection: if several claim it, the largest mask keeps it (a small far mask whose few
# LiDAR points bleed from a nearer car is the usual wrong claimant)
_lose = resolve_duplicate_gt({k: (v[0], v[3]) for k, v in assoc.items() if v[0] is not None and not v[2]})
for k in _lose:
    assoc[k] = (None, 'duplicate', False, assoc[k][3])
if _lose:
    print('Dropped ' + str(len(_lose)) + ' detection(s) that claimed a GT box already taken by a larger mask: ' +
          str(sorted(DET_ID[k] for k in _lose)))

rows = []
for mode_no, res in results_by_mode.items():
    for cam in CAMERAS:
        for r in res['objs'][cam]:
            key = (cam, r['prompt'], r['sam3_index'])
            gi, how, amb, _ = assoc[key]
            usable = gi is not None and not amb
            row = dict(id=DET_ID[key], cam=cam, cls=r['prompt'], mode=str(mode_no), label=r.get('o3_mode'),
                       scale=r.get('ssi_scale'),
                       shift_z=(None if r.get('ssi_shift') is None else float(r['ssi_shift'][2])),
                       obj_slope=(None if r.get('fit_ab') is None else float(r['fit_ab'][0])),
                       matched=usable, how=how)
            if usable:
                row.update(object_errors(r['obb_center'], r['obb_dims'], r['obb_yaw'], gt_boxes[gi]))
            rows.append(row)
n_obj = len(assoc)
n_amb = sum(1 for g, h, a_, _m in assoc.values() if g is not None and a_)
n_ok = sum(1 for g, h, a_, _m in assoc.values() if g is not None and not a_)
print(str(n_obj) + ' object(s): ' + str(n_ok) + ' matched to a GT box, ' + str(n_amb) + ' ambiguous (excluded), ' +
      str(n_obj - n_ok - n_amb) + ' without a match (excluded)')
diag = pd.DataFrame(rows)     # one row per (object, mode); also useful to inspect directly
pd.set_option('display.float_format', lambda v: '%.2f' % v)
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 60)

if diag.empty or not diag.matched.any():
    m = diag[diag.matched]
    print('No object could be matched to a GT box on this frame, so there is nothing to compare numerically.')
    print('Pick a frame/objects with more LiDAR on the masks (nearer cars), or set USE_MASK_OVERLAP_FALLBACK = True above.')
else:
    m = diag[diag.matched]
    print(str(m.id.nunique()) + ' object(s) matched to a GT box; unmatched (not compared): ' +
          str(sorted(set(diag[~diag.matched].id))))
    METRICS = ['center_err', 'range_err', 'lat_err', 'long_ratio', 'short_ratio', 'h_ratio', 'yaw_err', 'scale', 'shift_z', 'obj_slope']
    print('\n-- per object (rows) x mode (columns) --')
    MODE_KEYS = [str(x) for x in POINTMAP_MODES_TO_RUN]       # the real modes (the probe runs get their own table, 11.6)
    wide = m[m['mode'].isin(MODE_KEYS)].pivot_table(index=['id', 'cls', 'gt_range'], columns='mode', values=METRICS)
    display(wide.reindex(columns=METRICS, level=0).sort_index())

In [ ]:
if not diag.empty and diag.matched.any():
    MODE_KEYS = [str(x) for x in POINTMAP_MODES_TO_RUN]
    ids_per_mode = [set(m[m['mode'] == mo].id) for mo in MODE_KEYS]
    ok_ids = sorted(set.intersection(*ids_per_mode))
    both = m[m.id.isin(ok_ids) & m['mode'].isin(MODE_KEYS)]
    print('-- summary over the ' + str(len(ok_ids)) + ' object(s) matched in every mode '
          '(medians; errors: lower is better; ratios: closer to 1.0 is better) --')
    summ = both.groupby('mode').agg(
        n=('id', 'count'), center_err=('center_err', 'median'),
        abs_range_err=('range_err', lambda x: x.abs().median()), lat_err=('lat_err', 'median'),
        long_ratio=('long_ratio', 'median'), short_ratio=('short_ratio', 'median'), h_ratio=('h_ratio', 'median'),
        yaw_err=('yaw_err', 'median'), scale=('scale', 'median'), shift_z=('shift_z', 'median'))
    display(summ.reindex(MODE_KEYS))

    base = MODE_KEYS[0]
    for mo in MODE_KEYS[1:]:
        a = both[both['mode'] == base].set_index('id')
        b = both[both['mode'] == mo].set_index('id').loc[a.index]
        print('mode %s vs mode %s: closer to the GT center for %.0f%% of the objects (median center error %.2f vs %.2f m)'
              % (mo, base, 100 * (b.center_err < a.center_err).mean(), b.center_err.median(), a.center_err.median()))

    if ok_ids:
        fig, ax = plt.subplots(figsize=(max(6, 0.9 * len(ok_ids) + 2), 3.5))
        w = 0.8 / len(MODE_KEYS)
        for k, mo in enumerate(MODE_KEYS):
            vals = [both[(both.id == i) & (both['mode'] == mo)].center_err.iloc[0] for i in ok_ids]
            ax.bar(np.arange(len(ok_ids)) + k * w, vals, w, label='mode ' + mo)
        ax.set_xticks(np.arange(len(ok_ids)) + w * (len(MODE_KEYS) - 1) / 2)
        ax.set_xticklabels(['#' + str(i) for i in ok_ids])
        ax.set_ylabel('BEV center error [m]'); ax.set_title('per object, same GT box for every mode'); ax.legend()
        plt.tight_layout()

## 11.6 Token-drop probe — how much does SAM3D use the pointmap content?

`PROBE_MODE` was run normally and again with parts of its pointmap conditioning zeroed (Section 10). SAM3D still gets the
same shift and scale from the map; only the tokens the network *reads* change. Same objects, same seed, so any difference
comes from the zeroed tokens. All numbers are relative to the normal run of that mode.

| run | what is zeroed |
|---|---|
| `repeat` | nothing -- must be identical to normal (a determinism check; if not, differences below its size are noise) |
| `drop_both` | crop + full-image pointmap tokens (as in 10% of training samples) |
| `drop_full` | only the full-image stream (never dropped alone in training: read with care) |
| `drop_crop` | only the crop stream (same caveat) |

**How to read it.**
- `drop_both` ~ normal (small shifts, ratios ~1): the network barely reads the token content; only the shift and scale
  matter, so effort belongs in an accurate in-mask depth, not in densifying the map.
- `drop_both` much worse or very different, `drop_full` ~ normal, `drop_crop` ~ `drop_both`: the crop stream carries the
  information; the full-image background is irrelevant.
- `drop_full` clearly changes things: the full-image stream matters (accurate background depth is worth it).
- Which column moves says what depends on the tokens: range = depth, ratios = size, yaw = rotation.
With few objects this is indicative, not proof.

In [ ]:
if not RUN_TOKEN_PROBE:
    print('Token-drop probe is off (RUN_TOKEN_PROBE = False).')
else:
    base_label = PROBE_MODE
    labels = [base_label] + [str(PROBE_MODE) + ':' + v for v in PROBE_VARIANTS]

    def obj_index(res):
        return {(cam, r['prompt'], r['sam3_index']): r for cam in CAMERAS for r in res['objs'][cam]}

    idx = {lab: obj_index(results_by_mode[lab]) for lab in labels}
    keys = sorted(set.intersection(*[set(v) for v in idx.values()]))

    def yaw_diff(a, b):                     # heading difference, 0-90 deg (front/back flips ignored)
        d = (b - a) % np.pi
        return float(np.degrees(min(d, np.pi - d)))

    if not keys:
        print('No object was detected in every probe run.')
    else:
        rows_p = []
        for lab in labels[1:]:
            d_c, d_rng, r_long, r_h, d_yaw = [], [], [], [], []
            for k in keys:
                a, b = idx[base_label][k], idx[lab][k]
                ca, cb = np.asarray(a['obb_center'][:2]), np.asarray(b['obb_center'][:2])
                d_c.append(float(np.linalg.norm(cb - ca)))
                d_rng.append(float(np.hypot(*cb) - np.hypot(*ca)))
                r_long.append(max(b['obb_dims'][:2]) / max(a['obb_dims'][:2]))
                r_h.append(b['obb_dims'][2] / a['obb_dims'][2])
                d_yaw.append(yaw_diff(a['obb_yaw'], b['obb_yaw']))
            rows_p.append(dict(run=lab, n=len(keys), center_shift_m=np.median(d_c), max_center_shift_m=np.max(d_c),
                               range_shift_m=np.median(d_rng), long_ratio=np.median(r_long),
                               height_ratio=np.median(r_h), yaw_change_deg=np.median(d_yaw)))
        print('-- change relative to the normal run of mode ' + str(base_label) + ' (medians over ' + str(len(keys)) + ' object(s)) --')
        display(pd.DataFrame(rows_p).set_index('run'))
        rep = [r for r in rows_p if r['run'].endswith(':repeat')]
        if rep:
            if rep[0]['max_center_shift_m'] < 1e-2:
                print('Determinism check passed: the repeat run is identical, so the differences above come from the zeroed tokens.')
            else:
                print('WARNING: the repeat run differs by up to %.3f m -- treat differences of that size as noise.' % rep[0]['max_center_shift_m'])

        if not diag.empty and diag.matched.any():
            lab_str = [str(l) for l in labels]
            gt = diag[diag.matched & diag['mode'].isin(lab_str)]
            ok = set.intersection(*[set(gt[gt['mode'] == l].id) for l in lab_str]) if not gt.empty else set()
            if ok:
                print('\n-- error against GT (medians over the ' + str(len(ok)) + ' matched object(s) present in every run) --')
                g = gt[gt.id.isin(ok)].groupby('mode').agg(
                    center_err=('center_err', 'median'), abs_range_err=('range_err', lambda x: x.abs().median()),
                    lat_err=('lat_err', 'median'), long_ratio=('long_ratio', 'median'), short_ratio=('short_ratio', 'median'),
                    h_ratio=('h_ratio', 'median'), yaw_err=('yaw_err', 'median'))
                display(g.reindex(lab_str))
            else:
                print('\nNo object is matched to a GT box in every run, so no GT comparison is shown.')

## 12. 2D OBB visualization (one figure per mode)

No OBB output filter runs before this -- every detection (including whatever the pipeline gets wrong) is drawn,
on purpose. One panel per camera, one figure per mode.

In [ ]:
def draw_obb_on_image(img, corners, K, color, R_c2e=None, t_c2e=None, in_ego=False):
    pts = corners.astype(np.float64)
    if in_ego:
        pts = (R_c2e.T @ (pts - t_c2e).T).T  # ego -> camera
    uv = project(pts, K)
    out = img.copy()
    for i, j in _BBOX_EDGES:
        if uv[i, 0] < -9000 or uv[j, 0] < -9000:
            continue
        p1, p2 = tuple(uv[i].astype(int)), tuple(uv[j].astype(int))
        cv2.line(out, p1, p2, color, 2)
    return out


for mode_no in POINTMAP_MODES_TO_RUN:
    res = results_by_mode[mode_no]
    n_show = len(CAMERAS)
    fig, axes = plt.subplots(1, n_show, figsize=(7 * n_show, 5), squeeze=False)
    for ax, cam in zip(axes[0], CAMERAS):
        img_rgb_cam, _ = frames[cam].load_images()
        vis = img_rgb_cam.copy()
        for r in res['body'][cam]:
            vis = draw_obb_on_image(vis, r['obb_corners'], frames[cam].K, _PROMPT_COLORS['pedestrian'])
        for r in res['objs'][cam]:
            vis = draw_obb_on_image(vis, r['obb_corners'], frames[cam].K,
                                    _PROMPT_COLORS.get(r['prompt'], (200, 200, 200)),
                                    R_c2e=frames[cam].R_c2e, t_c2e=frames[cam].t_c2e, in_ego=True)
        ax.imshow(vis); ax.axis('off'); ax.set_title(cam)
    plt.suptitle(scene_name + '  frame ' + str(FRAME_IDX) + '  -- mode ' + str(mode_no) + ': ' +
                 POINTMAP_MODE_NAMES[mode_no] + ' -- 2D OBB reprojection')
    plt.tight_layout()
    plt.show()

## 13. BEV Visualization — one figure per mode

LiDAR colored by height, GT annotations as dashed boxes, pseudo-label OBBs as solid outlines colored by class, dark theme.
The LiDAR background is `pts_ego_full` (ground included). Pedestrians are identical in every figure; only the objects
differ between modes. No cross-camera duplicate suppression.

In [ ]:
# ── shared background: LiDAR + GT (GT boxes come from Section 11.5) ───────────
_rng  = np.random.default_rng(0)
_idx  = _rng.choice(len(pts_ego_full), min(40_000, len(pts_ego_full)), replace=False)
pts_d = pts_ego_full[_idx]
_mask = (np.abs(pts_d[:, 0]) < 60) & (np.abs(pts_d[:, 1]) < 30)
pts_d = pts_d[_mask]

base_traces = [go.Scatter(
    x=pts_d[:, 0], y=pts_d[:, 1], mode='markers',
    marker=dict(size=1.5, color=pts_d[:, 2], colorscale='Viridis',
               cmin=-2, cmax=3, showscale=True,
               colorbar=dict(title='Height (m)', thickness=12, x=1.01)),
    name='LiDAR', hoverinfo='skip',
)]
_GT_COLORS = {
    'vehicle': 'rgb(255,255,255)', 'human': 'rgb(255,210,100)',
    'movable_object': 'rgb(180,180,180)', 'static_object': 'rgb(140,140,140)',
}
_shown_gt = set()
for b in gt_boxes:
    top_cat = b['category'].split('.')[0]
    color   = _GT_COLORS.get(top_cat, 'rgb(200,200,200)')
    name    = 'GT ' + top_cat
    c       = b['corners_2d']
    base_traces.append(go.Scatter(
        x=[c[0, 0], c[1, 0], c[2, 0], c[3, 0], c[0, 0]],
        y=[c[0, 1], c[1, 1], c[2, 1], c[3, 1], c[0, 1]],
        mode='lines', line=dict(color=color, width=2, dash='dash'),
        name=name, legendgroup=name, showlegend=(name not in _shown_gt),
    ))
    _shown_gt.add(name)

_PSEUDO_COLORS = {
    'pedestrian': 'rgb(255,80,80)', 'car': 'rgb(80,220,80)',
    'bicycle': 'rgb(80,120,255)', 'motorcycle': 'rgb(255,160,0)',
    'bus': 'rgb(200,80,255)', 'truck': 'rgb(0,210,210)',
    'construction vehicle': 'rgb(255,80,180)', 'trailer': 'rgb(160,220,60)',
}


def _footprint_xy(corners_xy):
    cx, cy = corners_xy.mean(axis=0)
    angles = np.arctan2(corners_xy[:, 1] - cy, corners_xy[:, 0] - cx)
    return corners_xy[np.argsort(angles)]


def pseudo_traces(res):
    traces, shown = [], set()
    for cam in CAMERAS:
        for r in res['body'][cam]:
            corners_e = cam_to_ego(r['obb_corners'].astype(np.float64), frames[cam].R_c2e, frames[cam].t_c2e)
            foot = _footprint_xy(corners_e[:, :2])
            name = 'pseudo pedestrian'
            traces.append(go.Scatter(
                x=list(foot[:, 0]) + [foot[0, 0]], y=list(foot[:, 1]) + [foot[0, 1]],
                mode='lines', line=dict(color=_PSEUDO_COLORS['pedestrian'], width=2),
                name=name, legendgroup=name, showlegend=(name not in shown)))
            shown.add(name)
        for r in res['objs'][cam]:
            foot = _footprint_xy(np.asarray(r['obb_corners'])[:, :2])
            name = 'pseudo ' + r['prompt']
            traces.append(go.Scatter(
                x=list(foot[:, 0]) + [foot[0, 0]], y=list(foot[:, 1]) + [foot[0, 1]],
                mode='lines', line=dict(color=_PSEUDO_COLORS.get(r['prompt'], 'rgb(200,200,200)'), width=2),
                name=name, legendgroup=name, showlegend=(name not in shown)))
            shown.add(name)
    return traces


ego_marker = go.Scatter(
    x=[0], y=[0], mode='markers',
    marker=dict(symbol='arrow-up', angle=90, size=16, color='yellow', line=dict(color='black', width=1)),
    name='ego vehicle', showlegend=True)

for mode_no in POINTMAP_MODES_TO_RUN:
    fig_bev = go.Figure(data=base_traces + pseudo_traces(results_by_mode[mode_no]) + [ego_marker])
    fig_bev.update_layout(
        title=('BEV -- ' + scene_name + ', frame ' + str(FRAME_IDX) + ' -- mode ' + str(mode_no) + ': ' +
               POINTMAP_MODE_NAMES[mode_no] + ' | solid = pseudo labels | dashed = GT'),
        xaxis=dict(title='X ego (forward, m)', range=[-10, 60], scaleanchor='y', scaleratio=1,
                  showgrid=True, gridcolor='rgba(255,255,255,0.08)', zeroline=False),
        yaxis=dict(title='Y ego (left, m)', range=[-30, 30],
                  showgrid=True, gridcolor='rgba(255,255,255,0.08)', zeroline=False),
        plot_bgcolor='rgb(15,15,25)', paper_bgcolor='rgb(15,15,25)', font=dict(color='white'),
        legend=dict(bgcolor='rgba(0,0,0,0.5)', font=dict(size=11)),
        width=1000, height=580, margin=dict(l=0, r=80, t=40, b=0),
    )
    fig_bev.show()

## 14. Summary

In [ ]:
print(scene_name + ', frame ' + str(FRAME_IDX) + '  (dataset=' + DATASET + ')   modes run: ' + str(POINTMAP_MODES_TO_RUN))
for mode_no in POINTMAP_MODES_TO_RUN:
    res = results_by_mode[mode_no]
    print('\n mode ' + str(mode_no) + ' -- ' + POINTMAP_MODE_NAMES[mode_no])
    for cam in CAMERAS:
        print('  [' + cam + ']  pedestrians=' + str(len(res['body'][cam])) + '  objects=' + str(len(res['objs'][cam])))
        for r in res['objs'][cam]:
            print('      ' + format(r['prompt'], '<22') + ' label=' + str(r.get('o3_mode')))
print('\nThe per-object comparison table is the DataFrame `diag` (Section 11.5).')